# 27. Weight change and input sensitivity, measured together

### Why both numbers

Two different questions get asked about a model after unlearning.

**How far did the weights move?** The adapter plasticity study in
`reference/adapter-plasticity-study` answers this with one number it calls `rel_dW`:

    d_W = || W_eff - W0 || / || W0 ||

W0 is the weights before, W_eff is the weights now, and the bars mean the square root of
the sum of squared entries. If d_W is 0.05, the weights moved by 5 percent of their own
size.

**Does the model still work?** This study answers that with input sensitivity: how far any
single output score travels across the test set. A working model moves a lot, because the
picture decides the scores. A model that has stopped reading its input barely moves.

### Why they must both be read

This notebook exists because the two disagree, and the disagreement is the study's main
finding.

Over 100 unlearning steps the generated weights lose about 43 percent of their size. That
is an ordinary amount of movement. Over the same steps input sensitivity falls by a factor
of several million. A weight measurement alone would have shown nothing alarming at the
exact point where the model stopped working.

### What this notebook does

It applies one unlearning request to a saved model and records both numbers every 20
steps, on the task being removed and on the tasks being kept.

Nothing here is new science. Notebook 21 measured this already. What is new is that both
numbers now come from one shared module, `uncle/measure.py`, instead of from a function
copied into five notebooks in five versions that had drifted apart. Running this notebook
confirms the shared module reproduces what the separate copies produced.

### Cost

One unlearning request on a saved source model. A few minutes on A100, slower on CPU. It
trains no source model, writes nothing into the study, and cannot disturb a resume.

## 1. Load the code

This notebook runs from `main`, not from the study's pinned revision `65e2e7c8`.

That is deliberate and it is safe here. `uncle/measure.py` was written after the pin, so a
checkout at the pinned revision does not contain it. The pin exists to keep *training*
reproducible, and this notebook trains no source model: it loads saved weights, applies one
unlearning request with the study's own code, and measures. The unlearning loop itself is
checked against the pinned revision in section 5 before it runs.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
# The revision the study's results were produced at. Used as a check, not a checkout.
STUDY_CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-measure")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config, random_stream
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng
# The shared measurements. Both numbers come from here, and nowhere else.
from uncle.measure import probe, relative_change, weight_size

import torch
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("Study pinned revision:", STUDY_CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

Everything matches the saved study. `INTERVAL` only decides how often a measurement is
taken; it does not change the unlearning.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0                 # 0, 1 or 2
FORGET_TASK = "3"        # "3" or "14"

# Measure every this many unlearning steps. The study's own request is 100 steps,
# so 20 gives six readings including the one before anything happens.
INTERVAL = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Matching the study runner: no fast paths that change the arithmetic.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the saved model

The source model has learned the eight tasks of order 01 and forgotten nothing. It is the
model the study's own unlearning branches start from.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
if plan["code_version"] != STUDY_CODE_VERSION:
    raise ValueError("This study was produced at a different revision than this notebook expects.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=ORDER_TASKS, download=True)

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

# Forgetting protects every learned task except its own target. Copied from the
# study runner so this request matches the one the study makes.
PROTECTED = [t for t in state["seen"] if t != FORGET_TASK]
print("Source:", source_job["id"], "| removing task", FORGET_TASK,
      "| protecting", len(PROTECTED), "tasks")

## 5. Rebuild the model and check the unlearning loop against the pin

The loop below is the study's own `_forget_generated`, imported rather than copied. The
check confirms the file it comes from is byte for byte the version the study was produced
at, so running from `main` cannot have changed the unlearning.

In [ ]:
import hashlib

# The unlearning code must match the pinned revision exactly. If a later commit
# touched it, stop: the measurement would no longer describe the study's method.
for name in ("uncle/paired_diagnostic.py", "uncle/trainer.py", "uncle/hypernet.py"):
    now = subprocess.check_output(["git", "-C", str(REPO), "show", f"HEAD:{name}"])
    pinned = subprocess.check_output(["git", "-C", str(REPO), "show", f"{STUDY_CODE_VERSION}:{name}"])
    if hashlib.sha256(now).hexdigest() != hashlib.sha256(pinned).hexdigest():
        raise RuntimeError(f"{name} differs from the pinned revision. Do not trust this run.")
print("Unlearning code matches the pinned revision.")

from uncle.paired_diagnostic import _forget_generated

config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                   "requests": tuple(("learn", t) for t in ORDER_TASKS) + (("forget", FORGET_TASK),),
                   "burn_in": 100})
target = build_target(config)
hypernet = HyperNetwork(target, config)
trainer = UnCLe(hypernet, config, target, tasks, progress=True)
checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
restore_rng(state["all_rng"])
print("Model restored. Tasks learned:", list(state["seen"]))

## 6. Capture the reference weights

The relative weight change is measured against this moment, so it must be captured before
anything changes. The clone matters: `weights_for` returns tensors that are regenerated
each call, and without a copy the reference would follow the model as it changes.

The check that follows is the same one the adapter study makes at the start of its runs:
compared against itself, the change must be exactly zero.

In [ ]:
reference = {name: value.detach().clone()
             for name, value in hypernet.weights_for(FORGET_TASK).items()}

# Comparing the reference with itself must give exactly zero. A non-zero value
# here would mean the capture did not take a real copy.
check = relative_change(hypernet.weights_for(FORGET_TASK), reference)
if check != 0.0:
    raise RuntimeError(f"Reference capture is wrong: self-comparison gave {check}")
print(f"Reference captured: {len(reference)} tensors, "
      f"{sum(v.numel() for v in reference.values()):,} numbers. Self-check {check}.")

## 7. Run the unlearning request, measuring as it goes

One reading before anything happens, then one every `INTERVAL` steps.

Each reading records, for the task being removed: accuracy, how many classes the model ever
chooses, input sensitivity, the relative weight change against section 6, and the raw weight
size. It also records the mean accuracy of the tasks being kept, as a control: their normal
variation shows what an undisturbed task looks like over the same steps.

In [ ]:
import time

rows = []

def reading(step):
    """One full measurement. Both numbers, from the shared module."""
    forgotten = probe(hypernet, target, trainer, tasks[FORGET_TASK]["test"],
                      code_task=FORGET_TASK, classes=CLASSES, batch_size=EVAL_BATCH)
    kept = {t: probe(hypernet, target, trainer, tasks[t]["test"],
                     code_task=t, classes=CLASSES, batch_size=EVAL_BATCH)["accuracy"]
            for t in PROTECTED}
    row = {
        "step": step,
        "accuracy": forgotten["accuracy"],
        "distinct_classes": forgotten["distinct_classes"],
        "input_sensitivity": forgotten["input_sensitivity"],
        "rel_dW": relative_change(hypernet.weights_for(FORGET_TASK), reference),
        "weight_size": weight_size(hypernet, FORGET_TASK),
        "kept_mean": sum(kept.values()) / len(kept),
        "kept": kept,
    }
    rows.append(row)
    print(f"  step {step:>4} | task {FORGET_TASK} {row['accuracy']:5.1f}% "
          f"| classes {row['distinct_classes']:>2}/{CLASSES} "
          f"| sensitivity {row['input_sensitivity']:.3e} "
          f"| rel_dW {row['rel_dW']:.4f} "
          f"| kept {row['kept_mean']:5.1f}%")
    return row

print("Measuring. The first row is before any unlearning.\n")
began = time.perf_counter()
reading(0)

def on_step(record):
    """Called by the unlearning loop after each update."""
    step = record["step"]
    if step > 0 and step % INTERVAL == 0:
        reading(step)

_forget_generated(trainer, FORGET_TASK, PROTECTED, config.burn_in, on_step)
if rows[-1]["step"] != config.burn_in:
    reading(config.burn_in)
print(f"\nDone in {(time.perf_counter() - began) / 60:.1f} minutes.")

## 8. The comparison

The table puts the two measurements beside each other as a fraction of where they started.

This is the point of the notebook. Read the `weight_size` column against the
`input_sensitivity` column. If the weights move modestly while sensitivity collapses, a
weight measurement on its own would not have found the problem.

In [ ]:
start = rows[0]
print(f"{'step':>5}{'accuracy':>10}{'classes':>9}{'rel_dW':>9}"
      f"{'weight size':>13}{'of start':>10}{'sensitivity':>14}{'of start':>12}")
for row in rows:
    print(f"{row['step']:>5}{row['accuracy']:>9.1f}%{row['distinct_classes']:>9}"
          f"{row['rel_dW']:>9.4f}{row['weight_size']:>13.1f}"
          f"{100 * row['weight_size'] / start['weight_size']:>9.0f}%"
          f"{row['input_sensitivity']:>14.3e}"
          f"{row['input_sensitivity'] / start['input_sensitivity']:>12.2e}")

final = rows[-1]
weight_fraction = final["weight_size"] / start["weight_size"]
sensitivity_fraction = final["input_sensitivity"] / start["input_sensitivity"]
print(f"\nOver {final['step']} steps:")
print(f"  the weights moved {final['rel_dW']:.1%} of their own size")
print(f"  their size fell to {weight_fraction:.0%} of where it started")
print(f"  input sensitivity fell to {sensitivity_fraction:.2e} of where it started")
print(f"  the tasks being kept went from {start['kept_mean']:.1f}% to {final['kept_mean']:.1f}%")

if sensitivity_fraction < 1e-3 <= weight_fraction:
    print("\nThe two measurements disagree, which is the result this notebook is for.")
    print("The weights look ordinary. The model has stopped responding to its input.")
    print("Reporting only a weight measurement would have missed this.")
else:
    print("\nThe two measurements agree here. Record the numbers and say so.")

## 9. Save

One JSON file next to the study cache, holding every reading.

In [ ]:
import datetime

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out = CACHE / "measure" / f"weight_change_and_sensitivity_{stamp}.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({
    "notebook": "27_weight_change_and_sensitivity",
    "definition": "rel_dW = ||W_eff - W0|| / ||W0||, pooled, as in reference/adapter-plasticity-study",
    "code_revision": HEAD,
    "study_pinned_revision": STUDY_CODE_VERSION,
    "study_id": STUDY_ID, "order": ORDER, "seed": SEED, "forget_task": FORGET_TASK,
    "protected": PROTECTED, "interval": INTERVAL, "config": plan["config"],
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "rows": rows,
}, indent=2) + "\n", encoding="utf-8")
print("Saved:", out)

## 10. Using these measurements elsewhere

The two functions are now importable, so nothing needs copying into a new notebook:

```python
from uncle.measure import probe, relative_change, weight_size
```

`probe` returns accuracy, the per-class counts, how many classes the model ever chooses, and
input sensitivity. It switches to evaluation mode and restores the previous mode, which
matters: a BatchNorm layer in training mode rescales a nearly dead signal back to normal
size and hides the effect being measured.

`relative_change(current, reference)` is `rel_dW`. Pass `per_tensor=True` to also get the
same ratio for each tensor separately, which says where the change sits rather than only
how large it is.

For the adapter work that comes next, `relative_change` takes plain weight dictionaries, so
it applies unchanged to a LoRA adapter: pass the effective weights as `current` and the
frozen pretrained weights as `reference`.